# Stochastic Control for Asset Allocation

**Level:** Advanced  
**Estimated Time:** 150-180 minutes  
**Tags:** Stochastic Control, Dynamic Programming, HJB Equation, Merton Problem, Optimal Portfolio

## Overview

This notebook explores **stochastic optimal control theory** applied to dynamic asset allocation and consumption decisions. We solve classic problems in continuous-time finance using the **Hamilton-Jacobi-Bellman (HJB) equation** framework.

### What You'll Learn

By the end of this notebook, you will:

1. Understand stochastic control problems in continuous-time finance
2. Master the Hamilton-Jacobi-Bellman (HJB) equation and verification theorem
3. Solve Merton's optimal portfolio and consumption problem analytically
4. Implement numerical solutions for general utility functions
5. Analyze the impact of risk aversion, investment horizon, and constraints
6. Apply these techniques to real portfolio management problems

### Why Stochastic Control Matters

**In Modern Portfolio Management:**
- Dynamic rebalancing strategies that adapt to changing market conditions
- Optimal lifecycle investing for pension funds and endowments
- Risk management with state-dependent allocations
- Consumption smoothing for retirees
- Performance benchmarking for active managers

**Real-World Applications:**
- **Robo-advisors**: Automated portfolio rebalancing based on optimal control policies
- **Pension funds**: Lifecycle asset allocation glide paths
- **Hedge funds**: Dynamic leverage and market exposure management
- **Wealth management**: Consumption and bequest optimization
- **Central banks**: Optimal monetary policy under uncertainty

### Prerequisites

- **Mathematics**: Stochastic calculus (Ito's lemma), partial differential equations
- **Finance**: Portfolio theory, continuous-time finance
- **Python**: NumPy, SciPy optimization, numerical PDE solvers
- **Recommended**: Complete Black-Scholes and portfolio optimization notebooks first

### Key Concepts Covered

1. **Stochastic Control Framework**
   - Control variables vs state variables
   - Admissible controls and feedback policies
   - Value function and Bellman's principle
   
2. **Hamilton-Jacobi-Bellman Equation**
   - Derivation from dynamic programming
   - First-order conditions for optimality
   - Verification theorem
   
3. **Merton's Problem**
   - Power utility (CRRA)
   - Log utility (special case)
   - Analytic solutions
   
4. **Extensions and Generalizations**
   - Multiple risky assets
   - Transaction costs
   - Portfolio constraints
   - Labor income

---

## Mathematical Preliminaries

### Stochastic Differential Equations (SDEs)

We model asset prices using geometric Brownian motion:

$$
dS_t = \mu S_t dt + \sigma S_t dW_t
$$

where:
- $S_t$ = risky asset price at time $t$
- $\mu$ = expected return (drift)
- $\sigma$ = volatility
- $W_t$ = standard Brownian motion

### Ito's Lemma

For a function $f(t, S_t)$, Ito's lemma gives:

$$
df = \left(\frac{\partial f}{\partial t} + \mu S \frac{\partial f}{\partial S} + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 f}{\partial S^2}\right)dt + \sigma S \frac{\partial f}{\partial S} dW_t
$$

This is the cornerstone of continuous-time finance and stochastic control.

---

## References

1. **Merton, R.C.** (1971). "Optimal Consumption and Portfolio Rules in a Continuous-Time Model." *Journal of Economic Theory*, 3(4), 373-413.
2. **Shreve, S.E.** (2004). *Stochastic Calculus for Finance II: Continuous-Time Models*. Springer.
3. **Björk, T.** (2009). *Arbitrage Theory in Continuous Time*. Oxford University Press.
4. **Fleming, W.H., and Soner, H.M.** (2006). *Controlled Markov Processes and Viscosity Solutions*. Springer.

In [ ]:
# Import libraries\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom scipy import stats\nfrom scipy.optimize import minimize\nimport warnings\nwarnings.filterwarnings('ignore')\n\n# Configuration\nplt.style.use('seaborn-v0_8-darkgrid')\nsns.set_palette('husl')\n%matplotlib inline\nnp.random.seed(42)\n\nprint('Libraries imported successfully!')

## 1. Introduction to Stochastic Control

### What is Stochastic Control?

**Stochastic control theory** provides a mathematical framework for making optimal decisions over time under uncertainty. In finance, we use it to answer questions like:

- How much should I invest in risky vs safe assets?
- How should my portfolio allocation change as I age?
- What is the optimal consumption/savings tradeoff?
- How should I adjust my portfolio as wealth changes?

### Control Problem Setup

A stochastic control problem consists of:

1. **State Variable** $X_t$: The system state (e.g., wealth)
2. **Control Variable** $u_t$: The decision (e.g., portfolio allocation)
3. **Dynamics**: How state evolves under control:
   $$dX_t = f(t, X_t, u_t) dt + g(t, X_t, u_t) dW_t$$
4. **Objective**: Maximize expected utility:
   $$J(x, t) = \mathbb{E}\left[\int_t^T U(u_s, X_s) ds + \Psi(X_T)\right]$$

### The Value Function

The **value function** $V(t, x)$ is the maximum expected utility achievable from state $x$ at time $t$:

$$
V(t, x) = \sup_{u \in \mathcal{U}} \mathbb{E}\left[\int_t^T U(u_s, X_s) ds + \Psi(X_T) \mid X_t = x\right]
$$

where $\mathcal{U}$ is the set of admissible controls.

### Dynamic Programming Principle

Bellman's principle states that an optimal policy has the property:

> "Whatever the initial state and control are, the remaining decisions must constitute an optimal policy with regard to the state resulting from the first decision."

This leads to the **Bellman equation**, the foundation of dynamic programming.

### Example: Portfolio Allocation

Consider an investor with wealth $W_t$ who:
- Invests fraction $\pi_t$ in a risky stock with return $\mu$, volatility $\sigma$
- Invests fraction $(1-\pi_t)$ in a risk-free bond with return $r$
- Consumes at rate $c_t$

**Wealth dynamics:**
$$
dW_t = [(\mu - r)\pi_t W_t + rW_t - c_t] dt + \sigma \pi_t W_t dW_t
$$

**Objective:** Maximize expected utility from consumption:
$$
\max_{\pi, c} \mathbb{E}\left[\int_0^T U(c_t) dt + B(W_T)\right]
$$

This is **Merton's problem**, which we'll solve in detail below.

In [ ]:
# Merton's Optimal Portfolio and Consumption - Analytic Solution

def merton_optimal_allocation(mu, r, sigma, gamma):
    """
    Compute Merton's optimal portfolio allocation for CRRA utility.
    
    Parameters:
    -----------
    mu : float
        Expected return of risky asset
    r : float
        Risk-free rate
    sigma : float
        Volatility of risky asset
    gamma : float
        Coefficient of relative risk aversion
    
    Returns:
    --------
    pi_star : float
        Optimal fraction of wealth in risky asset
    """
    pi_star = (mu - r) / (gamma * sigma**2)
    return pi_star

def merton_consumption_rate(W, rho, gamma, r, mu, sigma):
    """
    Compute Merton's optimal consumption rate (infinite horizon).
    
    Parameters:
    -----------
    W : float or array
        Current wealth
    rho : float
        Subjective discount rate
    gamma : float
        Risk aversion coefficient
    r : float
        Risk-free rate
    mu : float
        Expected return of risky asset
    sigma : float
        Volatility of risky asset
    
    Returns:
    --------
    c_star : float or array
        Optimal consumption rate
    """
    # Effective rate of return on optimally invested portfolio
    sharpe_ratio_squared = ((mu - r) / sigma)**2
    portfolio_return = r + sharpe_ratio_squared * sigma**2 / (2 * gamma)
    
    # Propensity to consume
    consumption_propensity = rho - (1 - gamma) / gamma * portfolio_return
    
    c_star = consumption_propensity * W
    return c_star

def simulate_merton_wealth(W0, mu, r, sigma, gamma, rho, T, dt, n_paths=100):
    """
    Simulate wealth paths under Merton's optimal policy.
    
    Parameters:
    -----------
    W0 : float
        Initial wealth
    mu, r, sigma, gamma, rho : float
        Model parameters
    T : float
        Time horizon (years)
    dt : float
        Time step
    n_paths : int
        Number of simulation paths
    
    Returns:
    --------
    t : array
        Time grid
    W_paths : array
        Simulated wealth paths (n_paths x n_steps)
    c_paths : array
        Consumption paths
    """
    n_steps = int(T / dt)
    t = np.linspace(0, T, n_steps)
    
    # Optimal policies (constant for Merton)
    pi_star = merton_optimal_allocation(mu, r, sigma, gamma)
    
    # Initialize arrays
    W_paths = np.zeros((n_paths, n_steps))
    c_paths = np.zeros((n_paths, n_steps))
    W_paths[:, 0] = W0
    
    # Simulate paths
    for i in range(n_paths):
        for j in range(1, n_steps):
            W = W_paths[i, j-1]
            
            # Optimal consumption
            c = merton_consumption_rate(W, rho, gamma, r, mu, sigma)
            c_paths[i, j-1] = c
            
            # Wealth dynamics: dW = [π*W*(μ-r) + rW - c]dt + π*W*σ*dW
            drift = pi_star * W * (mu - r) + r * W - c
            diffusion = pi_star * W * sigma
            
            dW = drift * dt + diffusion * np.sqrt(dt) * np.random.normal()
            W_paths[i, j] = max(W + dW, 0)  # Prevent negative wealth
    
    # Final consumption
    c_paths[:, -1] = merton_consumption_rate(W_paths[:, -1], rho, gamma, r, mu, sigma)
    
    return t, W_paths, c_paths

# Example parameters (annual)
mu = 0.10      # Expected stock return: 10%
r = 0.03       # Risk-free rate: 3%
sigma = 0.20   # Stock volatility: 20%
gamma = 2.0    # Relative risk aversion
rho = 0.05     # Discount rate: 5%

# Compute optimal allocation
pi_star = merton_optimal_allocation(mu, r, sigma, gamma)

print("=" * 70)
print("MERTON'S OPTIMAL PORTFOLIO ALLOCATION")
print("=" * 70)
print(f"\nMarket Parameters:")
print(f"  Expected stock return (μ): {mu*100:.1f}%")
print(f"  Risk-free rate (r):        {r*100:.1f}%")
print(f"  Stock volatility (σ):      {sigma*100:.1f}%")
print(f"  Sharpe ratio:              {(mu-r)/sigma:.3f}")
print(f"\nInvestor Parameters:")
print(f"  Risk aversion (γ):         {gamma:.1f}")
print(f"  Discount rate (ρ):         {rho*100:.1f}%")
print(f"\nOptimal Policy:")
print(f"  Risky asset allocation (π*): {pi_star*100:.2f}%")
print(f"  Risk-free allocation:        {(1-pi_star)*100:.2f}%")

# Show how allocation varies with risk aversion
print("\n" + "=" * 70)
print("ALLOCATION VS RISK AVERSION")
print("=" * 70)
gammas = np.array([0.5, 1.0, 2.0, 3.0, 5.0, 10.0])
allocations = [(g, merton_optimal_allocation(mu, r, sigma, g)) for g in gammas]

print(f"\n{'Risk Aversion (γ)':<20} {'Risky Allocation (π*)':<25} {'Leverage?'}")
print("-" * 70)
for g, pi in allocations:
    leverage = "YES (Borrowing)" if pi > 1.0 else "NO"
    print(f"{g:<20.1f} {pi*100:<24.2f}% {leverage}")

# Wealth simulation example
W0 = 100000  # Initial wealth: $100k
print(f"\n\nFor initial wealth W₀ = ${W0:,.0f}:")
print(f"  Investment in risky asset:  ${pi_star * W0:,.0f}")
print(f"  Investment in risk-free:    ${(1-pi_star) * W0:,.0f}")
print(f"  Optimal consumption rate:   ${merton_consumption_rate(W0, rho, gamma, r, mu, sigma):,.0f}/year")

# Run simulation
print("\n\nSimulating 100 wealth paths over 30 years...")
t, W_paths, c_paths = simulate_merton_wealth(W0, mu, r, sigma, gamma, rho, T=30, dt=0.1, n_paths=100)
print("Simulation complete!")

In [ ]:
# Comprehensive visualization of Merton's solution

print("=" * 70)
print("COMPREHENSIVE VISUALIZATION OF OPTIMAL POLICIES")
print("=" * 70)

# Create massive visualization grid
fig = plt.figure(figsize=(22, 26))
gs = fig.add_gridspec(5, 3, hspace=0.35, wspace=0.3)

# 1. Wealth Paths Simulation (Multiple Scenarios)
ax1 = fig.add_subplot(gs[0, :2])
n_paths_plot = 20
for i in range(n_paths_plot):
    alpha_val = 0.3 if i < n_paths_plot else 0.8
    linewidth = 0.8 if i < n_paths_plot else 2
    ax1.plot(t, W_paths[i], alpha=alpha_val, linewidth=linewidth, color='#3498db')

# Add mean wealth path
mean_wealth = W_paths.mean(axis=0)
ax1.plot(t, mean_wealth, linewidth=3.5, color='red', 
        label=f'Mean Wealth', linestyle='--')

# Add confidence bands
wealth_p10 = np.percentile(W_paths, 10, axis=0)
wealth_p90 = np.percentile(W_paths, 90, axis=0)
ax1.fill_between(t, wealth_p10, wealth_p90, alpha=0.2, color='#3498db', 
                 label='10th-90th percentile')

ax1.axhline(y=W0, color='green', linestyle=':', linewidth=2, 
           label=f'Initial Wealth: ${W0:,.0f}')
ax1.set_xlabel('Time (years)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Wealth ($)', fontsize=13, fontweight='bold')
ax1.set_title(f'Simulated Wealth Paths Under Merton\'s Optimal Policy\n(γ={gamma}, μ={mu*100:.0f}%, σ={sigma*100:.0f}%)', 
             fontsize=15, fontweight='bold')
ax1.legend(loc='upper left', fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 30)

# 2. Terminal Wealth Distribution
ax2 = fig.add_subplot(gs[0, 2])
terminal_wealth = W_paths[:, -1]
ax2.hist(terminal_wealth, bins=40, color='#2ecc71', alpha=0.7, edgecolor='black', density=True)

# Add statistics
median_terminal = np.median(terminal_wealth)
mean_terminal = np.mean(terminal_wealth)
ax2.axvline(x=median_terminal, color='blue', linestyle='--', linewidth=2.5, 
           label=f'Median: ${median_terminal:,.0f}')
ax2.axvline(x=mean_terminal, color='red', linestyle='--', linewidth=2.5, 
           label=f'Mean: ${mean_terminal:,.0f}')
ax2.axvline(x=W0, color='green', linestyle=':', linewidth=2, 
           label=f'Initial: ${W0:,.0f}')

ax2.set_xlabel('Terminal Wealth ($)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Density', fontsize=12, fontweight='bold')
ax2.set_title(f'Terminal Wealth Distribution\n(After {int(t[-1])} Years)', 
             fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

# 3. Consumption Paths
ax3 = fig.add_subplot(gs[1, :2])
n_paths_consumption = 15
for i in range(n_paths_consumption):
    ax3.plot(t[:-1], c_paths[i, :-1], alpha=0.4, linewidth=0.8, color='#e74c3c')

# Mean consumption
mean_consumption = c_paths.mean(axis=0)
ax3.plot(t[:-1], mean_consumption[:-1], linewidth=3.5, color='darkred', 
        label='Mean Consumption', linestyle='--')

# Add consumption as % of wealth
ax3_twin = ax3.twinx()
consumption_rate = mean_consumption / (mean_wealth + 1e-10)
ax3_twin.plot(t[:-1], consumption_rate[:-1] * 100, linewidth=2.5, 
             color='purple', linestyle=':', label='Consumption Rate (%)', alpha=0.8)
ax3_twin.set_ylabel('Consumption / Wealth (%)', fontsize=12, fontweight='bold', color='purple')
ax3_twin.tick_params(axis='y', labelcolor='purple')

ax3.set_xlabel('Time (years)', fontsize=13, fontweight='bold')
ax3.set_ylabel('Consumption ($/year)', fontsize=13, fontweight='bold')
ax3.set_title('Consumption Paths Under Optimal Policy', 
             fontsize=15, fontweight='bold')
ax3.legend(loc='upper left', fontsize=11)
ax3.grid(True, alpha=0.3)
ax3.set_xlim(0, 30)

# 4. Portfolio Allocation vs Risk Aversion
ax4 = fig.add_subplot(gs[1, 2])
gammas_range = np.linspace(0.5, 10, 50)
allocations = [merton_optimal_allocation(mu, r, sigma, g) * 100 for g in gammas_range]

ax4.plot(gammas_range, allocations, linewidth=3, color='#3498db')
ax4.fill_between(gammas_range, 0, allocations, alpha=0.3, color='#3498db')

# Mark some key points
for g_mark in [1, 2, 3, 5]:
    alloc_mark = merton_optimal_allocation(mu, r, sigma, g_mark) * 100
    ax4.plot(g_mark, alloc_mark, 'ro', markersize=10)
    ax4.annotate(f'γ={g_mark}\n{alloc_mark:.1f}%', 
                xy=(g_mark, alloc_mark), xytext=(10, 10),
                textcoords='offset points', fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7))

ax4.axhline(y=100, color='red', linestyle='--', linewidth=2, alpha=0.5, label='100% (No Leverage)')
ax4.axhline(y=0, color='green', linestyle='--', linewidth=2, alpha=0.5)

ax4.set_xlabel('Risk Aversion (γ)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Risky Asset Allocation (%)', fontsize=12, fontweight='bold')
ax4.set_title('Optimal Allocation vs Risk Aversion\n(Merton\'s Formula)', 
             fontsize=14, fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)
ax4.set_xlim(0.5, 10)

# 5. Wealth vs Consumption Phase Diagram
ax5 = fig.add_subplot(gs[2, 0])
# Take a single representative path
path_idx = 50
wealth_sample = W_paths[path_idx, ::10]  # Sample every 10th point
consumption_sample = c_paths[path_idx, ::10]

scatter = ax5.scatter(wealth_sample, consumption_sample, 
                      c=np.arange(len(wealth_sample)), cmap='viridis', 
                      s=100, alpha=0.7, edgecolors='black', linewidth=1.5)

# Add trend line
z = np.polyfit(wealth_sample, consumption_sample, 1)
p = np.poly1d(z)
wealth_fit = np.linspace(wealth_sample.min(), wealth_sample.max(), 100)
ax5.plot(wealth_fit, p(wealth_fit), "r--", linewidth=2.5, alpha=0.8, 
        label=f'Linear fit: C = {z[0]:.3f}W + {z[1]:.1f}')

ax5.set_xlabel('Wealth ($)', fontsize=12, fontweight='bold')
ax5.set_ylabel('Consumption ($/year)', fontsize=12, fontweight='bold')
ax5.set_title('Wealth-Consumption Relationship\n(Color = Time)', 
             fontsize=14, fontweight='bold')
ax5.legend(fontsize=10)
ax5.grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=ax5)
cbar.set_label('Time (years)', fontsize=10)

# 6. Sensitivity to Expected Return (μ)
ax6 = fig.add_subplot(gs[2, 1])
mu_range = np.linspace(0.05, 0.20, 50)
allocations_mu = [merton_optimal_allocation(m, r, sigma, gamma) * 100 for m in mu_range]

ax6.plot(mu_range * 100, allocations_mu, linewidth=3, color='#2ecc71')
ax6.fill_between(mu_range * 100, 0, allocations_mu, alpha=0.3, color='#2ecc71')
ax6.axvline(x=mu * 100, color='red', linestyle='--', linewidth=2, 
           label=f'Base case: μ={mu*100:.0f}%')

ax6.set_xlabel('Expected Return μ (%)', fontsize=12, fontweight='bold')
ax6.set_ylabel('Risky Asset Allocation (%)', fontsize=12, fontweight='bold')
ax6.set_title('Allocation Sensitivity to Expected Return\n(Fixed γ={})'.format(gamma), 
             fontsize=14, fontweight='bold')
ax6.legend(fontsize=10)
ax6.grid(True, alpha=0.3)

# 7. Sensitivity to Volatility (σ)
ax7 = fig.add_subplot(gs[2, 2])
sigma_range = np.linspace(0.10, 0.40, 50)
allocations_sigma = [merton_optimal_allocation(mu, r, sig, gamma) * 100 for sig in sigma_range]

ax7.plot(sigma_range * 100, allocations_sigma, linewidth=3, color='#e74c3c')
ax7.fill_between(sigma_range * 100, 0, allocations_sigma, alpha=0.3, color='#e74c3c')
ax7.axvline(x=sigma * 100, color='blue', linestyle='--', linewidth=2, 
           label=f'Base case: σ={sigma*100:.0f}%')

ax7.set_xlabel('Volatility σ (%)', fontsize=12, fontweight='bold')
ax7.set_ylabel('Risky Asset Allocation (%)', fontsize=12, fontweight='bold')
ax7.set_title('Allocation Sensitivity to Volatility\n(Fixed γ={})'.format(gamma), 
             fontsize=14, fontweight='bold')
ax7.legend(fontsize=10)
ax7.grid(True, alpha=0.3)

# 8. Wealth Percentiles Over Time
ax8 = fig.add_subplot(gs[3, :2])
percentiles = [5, 25, 50, 75, 95]
colors_percentiles = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

for p, color in zip(percentiles, colors_percentiles):
    wealth_p = np.percentile(W_paths, p, axis=0)
    ax8.plot(t, wealth_p, linewidth=2.5, label=f'{p}th percentile', color=color)

ax8.axhline(y=W0, color='black', linestyle=':', linewidth=2, alpha=0.5, 
           label=f'Initial: ${W0:,.0f}')
ax8.set_xlabel('Time (years)', fontsize=13, fontweight='bold')
ax8.set_ylabel('Wealth ($)', fontsize=13, fontweight='bold')
ax8.set_title('Wealth Percentiles Over Time\n(Distribution Evolution)', 
             fontsize=15, fontweight='bold')
ax8.legend(loc='upper left', fontsize=11, ncol=2)
ax8.grid(True, alpha=0.3)
ax8.set_xlim(0, 30)

# 9. Sharpe Ratio Impact
ax9 = fig.add_subplot(gs[3, 2])
sharpe_range = np.linspace(0.1, 1.0, 50)
# For each Sharpe ratio, calculate implied mu and allocation
allocations_sharpe = []
for sr in sharpe_range:
    mu_implied = r + sr * sigma
    alloc = merton_optimal_allocation(mu_implied, r, sigma, gamma) * 100
    allocations_sharpe.append(alloc)

ax9.plot(sharpe_range, allocations_sharpe, linewidth=3, color='#9467bd')
ax9.fill_between(sharpe_range, 0, allocations_sharpe, alpha=0.3, color='#9467bd')
current_sharpe = (mu - r) / sigma
ax9.axvline(x=current_sharpe, color='red', linestyle='--', linewidth=2, 
           label=f'Current SR: {current_sharpe:.2f}')

ax9.set_xlabel('Sharpe Ratio', fontsize=12, fontweight='bold')
ax9.set_ylabel('Risky Asset Allocation (%)', fontsize=12, fontweight='bold')
ax9.set_title('Allocation vs Sharpe Ratio\n(Risk-Adjusted Returns)', 
             fontsize=14, fontweight='bold')
ax9.legend(fontsize=10)
ax9.grid(True, alpha=0.3)

# 10. Comparison: Different Risk Aversions
ax10 = fig.add_subplot(gs[4, :])
gammas_compare = [0.5, 1.0, 2.0, 5.0]
colors_gamma = ['#e74c3c', '#f39c12', '#3498db', '#9b59b6']

for g, color in zip(gammas_compare, colors_gamma):
    # Simulate one path for each gamma
    np.random.seed(42)
    W_single = np.zeros(len(t))
    W_single[0] = W0
    pi_g = merton_optimal_allocation(mu, r, sigma, g)
    
    for i in range(1, len(t)):
        W = W_single[i-1]
        c = merton_consumption_rate(W, rho, g, r, mu, sigma)
        drift = pi_g * W * (mu - r) + r * W - c
        diffusion = pi_g * W * sigma
        dW = drift * 0.1 + diffusion * np.sqrt(0.1) * np.random.normal()
        W_single[i] = max(W + dW, 0)
    
    ax10.plot(t, W_single, linewidth=2.5, label=f'γ={g} (π*={pi_g*100:.1f}%)', 
             color=color)

ax10.axhline(y=W0, color='black', linestyle=':', linewidth=2, alpha=0.5, 
            label=f'Initial: ${W0:,.0f}')
ax10.set_xlabel('Time (years)', fontsize=13, fontweight='bold')
ax10.set_ylabel('Wealth ($)', fontsize=13, fontweight='bold')
ax10.set_title('Wealth Paths for Different Risk Aversions\n(Same Random Shocks)', 
              fontsize=15, fontweight='bold')
ax10.legend(loc='best', fontsize=11, ncol=2)
ax10.grid(True, alpha=0.3)
ax10.set_xlim(0, 30)

plt.suptitle('COMPREHENSIVE VISUALIZATION OF MERTON\'S OPTIMAL POLICIES', 
            fontsize=20, fontweight='bold', y=0.998)

plt.tight_layout()
plt.show()

# Print detailed statistics
print("\n" + "=" * 70)
print("DETAILED STATISTICS FROM SIMULATIONS")
print("=" * 70)

print(f"\n1. TERMINAL WEALTH (after {int(t[-1])} years):")
print(f"   Mean:              ${mean_terminal:,.0f}")
print(f"   Median:            ${median_terminal:,.0f}")
print(f"   Std Dev:           ${np.std(terminal_wealth):,.0f}")
print(f"   5th percentile:    ${np.percentile(terminal_wealth, 5):,.0f}")
print(f"   95th percentile:   ${np.percentile(terminal_wealth, 95):,.0f}")
print(f"   Min:               ${terminal_wealth.min():,.0f}")
print(f"   Max:               ${terminal_wealth.max():,.0f}")

growth_rate = (mean_terminal / W0) ** (1/t[-1]) - 1
print(f"\n2. WEALTH GROWTH:")
print(f"   Average annual growth: {growth_rate*100:.2f}%")
print(f"   Total return (mean):   {(mean_terminal/W0 - 1)*100:.2f}%")

print(f"\n3. CONSUMPTION STATISTICS:")
print(f"   Mean consumption:      ${mean_consumption.mean():,.0f}/year")
print(f"   Initial consumption:   ${mean_consumption[0]:,.0f}/year")
print(f"   Final consumption:     ${mean_consumption[-1]:,.0f}/year")
print(f"   Consumption growth:    {(mean_consumption[-1]/mean_consumption[0] - 1)*100:.1f}%")

print(f"\n4. PORTFOLIO ALLOCATION (Merton's Formula):")
print(f"   Risky asset:           {pi_star*100:.2f}%")
print(f"   Risk-free asset:       {(1-pi_star)*100:.2f}%")
if pi_star > 1:
    print(f"   Leverage:              {(pi_star-1)*100:.2f}% (borrowing at risk-free rate)")

print(f"\n5. KEY INSIGHTS:")
print(f"   - Higher risk aversion → Lower equity allocation")
print(f"   - Allocation π* ∝ (μ-r) / (γσ²)")
print(f"   - Optimal policy is CONSTANT over time (myopic)")
print(f"   - Consumption is proportional to wealth")

wealth_cv = np.std(terminal_wealth) / mean_terminal
print(f"\n6. RISK METRICS:")
print(f"   Coefficient of variation: {wealth_cv:.2f}")
print(f"   Probability of loss:      {(terminal_wealth < W0).mean()*100:.1f}%")
print(f"   Probability of doubling:  {(terminal_wealth > 2*W0).mean()*100:.1f}%")

print("\n" + "=" * 70)
print("Comprehensive visualization complete!")
print("=" * 70)

## 4. Comprehensive Visualization of Optimal Policies

### Understanding the Results Through Visualization

Visualizations are crucial for understanding stochastic control solutions:

1. **Wealth Paths**: How does wealth evolve under optimal control?
2. **Consumption Patterns**: When and how much should we consume?
3. **Portfolio Allocation**: How does risk aversion affect asset allocation?
4. **Sensitivity Analysis**: Impact of parameters on optimal policies
5. **Lifecycle Patterns**: How policies change over time

These visualizations help answer practical questions:
- Should I be more aggressive when young?
- How does risk aversion impact my portfolio?
- What happens in market downturns?
- How should I adjust consumption as wealth changes?

Let's create publication-quality visualizations to explore these dynamics!

## 5. Extensions and Advanced Topics

### Beyond the Basic Merton Problem

Real-world portfolio management requires extensions to handle:

1. **Transaction Costs**: Friction in rebalancing
2. **Portfolio Constraints**: No-short-selling, leverage limits
3. **Multiple Risky Assets**: Diversification opportunities  
4. **Labor Income**: Human capital as an asset
5. **Stochastic Parameters**: Time-varying μ, σ, r

**Transaction Costs:**

With proportional transaction costs $\lambda$, the HJB equation becomes:

$$V_t + \max_{\pi, c} \left\{U(c) + \mathcal{L}^{\pi,c}V - \lambda |d\pi| W V_W\right\} = 0$$

This leads to a **no-trade region**: only rebalance when deviation exceeds a threshold.

**Portfolio Constraints:**

With constraint $\pi \in [\pi_{min}, \pi_{max}]$:

$$\pi^* = \max\left(\pi_{min}, \min\left(\frac{\mu-r}{\gamma \sigma^2}, \pi_{max}\right)\right)$$

**Multiple Assets:**

For $n$ risky assets with returns $\mathbf{dS}/\mathbf{S} = \boldsymbol{\mu}dt + \boldsymbol{\Sigma}^{1/2}d\mathbf{W}$:

$$\boldsymbol{\pi}^* = \frac{1}{\gamma}\boldsymbol{\Sigma}^{-1}(\boldsymbol{\mu} - r\mathbf{1})$$

Let's explore these extensions through examples and visualizations!

## 6. Lifecycle Investing: Age-Dependent Asset Allocation

### The Lifecycle Portfolio Problem

**Key Question**: Should young investors be more aggressive than old investors?

**Traditional Advice ("100 - age" rule)**:
- Age 30 → 70% stocks
- Age 60 → 40% stocks

**Merton's Result**: Optimal allocation is **constant** (independent of age/wealth)!

**Why the discrepancy?**

Merton's model misses key features:
1. **Human capital**: Young workers have valuable future labor income
2. **Flexibility**: Young have more time to recover from losses
3. **Changing risk aversion**: Risk tolerance may decrease with age
4. **Finite horizon**: Approaching retirement changes objectives

**Lifecycle Model with Human Capital:**

Total wealth = Financial wealth + Present value of labor income

$$W_{total} = W_{financial} + PV(Labor Income)$$

Young investors:
- Large human capital (bond-like asset)
- Should invest MORE in stocks to balance portfolio
- Can recover from losses via future earnings

**Let's visualize lifecycle asset allocation patterns!**

In [ ]:
# Lifecycle asset allocation glide path

print("=" * 70)
print("LIFECYCLE INVESTING: AGE-DEPENDENT ASSET ALLOCATION")
print("=" * 70)

# Lifecycle parameters
ages = np.arange(25, 70, 1)  # Ages 25 to 69
retirement_age = 65
years_to_retirement = retirement_age - ages

# Model 1: Simple "100 - age" rule
allocation_simple = np.maximum(0, (100 - ages) / 100)

# Model 2: Time-varying risk aversion (increases with age)
# γ(age) = γ_young + (γ_old - γ_young) * (age - 25) / (65 - 25)
gamma_young = 1.5
gamma_old = 4.0
gamma_lifecycle = gamma_young + (gamma_old - gamma_young) * (ages - 25) / (retirement_age - 25)
allocation_time_varying_gamma = np.array([merton_optimal_allocation(mu, r, sigma, g) 
                                          for g in gamma_lifecycle])

# Model 3: With human capital
# Assume annual salary = $80k, growing at 2%/year until retirement
salary = 80000
salary_growth = 0.02
discount_rate = 0.04  # Discount rate for PV calculation

human_capital = []
for age in ages:
    years_left = max(0, retirement_age - age)
    # PV of future labor income
    hc = sum([salary * (1 + salary_growth)**t / (1 + discount_rate)**t 
             for t in range(years_left)])
    human_capital.append(hc)

human_capital = np.array(human_capital)

# Financial wealth assumption (accumulates over time)
financial_wealth = np.linspace(50000, 500000, len(ages))

# Total wealth
total_wealth = financial_wealth + human_capital

# Human capital is bond-like, so to maintain target allocation:
# If human capital is X% of total wealth, increase stock allocation
hc_fraction = human_capital / total_wealth
target_equity_ratio = 0.6  # Target 60% of total wealth in equities

# Allocation in financial portfolio to achieve target
allocation_with_hc = target_equity_ratio + (target_equity_ratio * hc_fraction)
allocation_with_hc = np.minimum(allocation_with_hc, 1.5)  # Cap at 150%

# Model 4: Target Date Fund (typical glide path)
# Aggressive when young, conservative near retirement
allocation_target_date = np.zeros(len(ages))
for i, age in enumerate(ages):
    if age < 40:
        allocation_target_date[i] = 0.90  # 90% stocks
    elif age < 50:
        allocation_target_date[i] = 0.85
    elif age < 60:
        allocation_target_date[i] = 0.70
    else:
        allocation_target_date[i] = 0.50  # 50% stocks at retirement

# Create comprehensive lifecycle visualization
fig, axes = plt.subplots(3, 2, figsize=(18, 18))

# 1. Glide Paths Comparison
ax = axes[0, 0]
ax.plot(ages, allocation_simple * 100, linewidth=3, label='"100 - Age" Rule', 
       color='#3498db', marker='o', markersize=4, markevery=5)
ax.plot(ages, allocation_time_varying_gamma * 100, linewidth=3, 
       label='Time-Varying Risk Aversion', color='#e74c3c', marker='s', 
       markersize=4, markevery=5)
ax.plot(ages, allocation_with_hc * 100, linewidth=3, 
       label='With Human Capital', color='#2ecc71', marker='^', 
       markersize=4, markevery=5)
ax.plot(ages, allocation_target_date * 100, linewidth=3, 
       label='Target Date Fund', color='#9b59b6', marker='d', 
       markersize=4, markevery=5)

ax.axvline(x=retirement_age, color='red', linestyle='--', linewidth=2, alpha=0.5, 
          label=f'Retirement Age: {retirement_age}')
ax.axhline(y=100, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.axhline(y=50, color='gray', linestyle=':', linewidth=1, alpha=0.5)

ax.set_xlabel('Age (years)', fontsize=13, fontweight='bold')
ax.set_ylabel('Equity Allocation (%)', fontsize=13, fontweight='bold')
ax.set_title('Lifecycle Asset Allocation Glide Paths\n(Different Models)', 
            fontsize=15, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 160)

# 2. Human Capital vs Financial Wealth
ax = axes[0, 1]
ax.fill_between(ages, 0, financial_wealth / 1000, alpha=0.7, 
               label='Financial Wealth', color='#3498db')
ax.fill_between(ages, financial_wealth / 1000, total_wealth / 1000, 
               alpha=0.7, label='Human Capital', color='#2ecc71')
ax.plot(ages, total_wealth / 1000, linewidth=3, color='black', 
       label='Total Wealth', linestyle='--')

ax.set_xlabel('Age (years)', fontsize=13, fontweight='bold')
ax.set_ylabel('Wealth ($1000s)', fontsize=13, fontweight='bold')
ax.set_title('Human Capital vs Financial Wealth\nOver Lifecycle', 
            fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# 3. Risk Aversion Over Lifecycle
ax = axes[1, 0]
ax.plot(ages, gamma_lifecycle, linewidth=3.5, color='#e74c3c', marker='o', 
       markersize=6, markevery=5)
ax.axhline(y=gamma_young, color='blue', linestyle='--', linewidth=2, 
          label=f'Young (γ={gamma_young})')
ax.axhline(y=gamma_old, color='purple', linestyle='--', linewidth=2, 
          label=f'Old (γ={gamma_old})')
ax.axvline(x=retirement_age, color='red', linestyle=':', linewidth=2, alpha=0.5)

ax.set_xlabel('Age (years)', fontsize=13, fontweight='bold')
ax.set_ylabel('Risk Aversion (γ)', fontsize=13, fontweight='bold')
ax.set_title('Time-Varying Risk Aversion\n(Increases with Age)', 
            fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# 4. Human Capital as % of Total Wealth
ax = axes[1, 1]
ax.plot(ages, hc_fraction * 100, linewidth=3.5, color='#2ecc71', marker='s', 
       markersize=6, markevery=5)
ax.fill_between(ages, 0, hc_fraction * 100, alpha=0.3, color='#2ecc71')
ax.axvline(x=retirement_age, color='red', linestyle='--', linewidth=2, 
          alpha=0.5, label='Retirement')

ax.set_xlabel('Age (years)', fontsize=13, fontweight='bold')
ax.set_ylabel('Human Capital (% of Total Wealth)', fontsize=13, fontweight='bold')
ax.set_title('Human Capital Proportion\n(Declines with Age)', 
            fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# 5. Wealth Accumulation by Strategy
ax = axes[2, 0]
# Simulate wealth accumulation for different strategies
np.random.seed(42)
T_lifecycle = retirement_age - 25
n_years = len(ages)

# Simple simulation for different strategies
wealth_paths = {}
for strategy, alloc in [
    ('Simple Rule', allocation_simple),
    ('Time-Varying γ', allocation_time_varying_gamma),
    ('With Human Capital', allocation_with_hc),
    ('Target Date', allocation_target_date)
]:
    W_sim = np.zeros(n_years)
    W_sim[0] = 50000  # Starting wealth at age 25
    
    for i in range(1, n_years):
        W = W_sim[i-1]
        pi = alloc[i-1]
        
        # Simple wealth evolution (no consumption for this simulation)
        portfolio_return = pi * (mu - r) + r
        annual_return = np.exp((portfolio_return - 0.5 * (pi * sigma)**2) + 
                               pi * sigma * np.random.normal())
        
        # Add annual contribution
        contribution = 10000  # $10k annual contribution
        W_sim[i] = W * annual_return + contribution
    
    wealth_paths[strategy] = W_sim
    ax.plot(ages, W_sim / 1000, linewidth=3, label=strategy, marker='o', 
           markersize=4, markevery=10)

ax.axvline(x=retirement_age, color='red', linestyle='--', linewidth=2, 
          alpha=0.5, label='Retirement')
ax.set_xlabel('Age (years)', fontsize=13, fontweight='bold')
ax.set_ylabel('Wealth ($1000s)', fontsize=13, fontweight='bold')
ax.set_title('Wealth Accumulation by Strategy\n(with $10k annual contributions)', 
            fontsize=15, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 6. Years to Retirement vs Allocation
ax = axes[2, 1]
years_left = retirement_age - ages
ax.scatter(years_left, allocation_with_hc * 100, s=100, alpha=0.7, 
          c=ages, cmap='viridis', edgecolors='black', linewidth=1.5)

# Add trend line
z = np.polyfit(years_left, allocation_with_hc * 100, 2)
p = np.poly1d(z)
years_fit = np.linspace(0, 40, 100)
ax.plot(years_fit, p(years_fit), 'r--', linewidth=3, alpha=0.8, 
       label='Quadratic fit')

ax.set_xlabel('Years Until Retirement', fontsize=13, fontweight='bold')
ax.set_ylabel('Equity Allocation (%)', fontsize=13, fontweight='bold')
ax.set_title('Allocation vs Time Horizon\n(With Human Capital Model)', 
            fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
cbar = plt.colorbar(ax.collections[0], ax=ax)
cbar.set_label('Age (years)', fontsize=10)

plt.suptitle('LIFECYCLE INVESTING AND ASSET ALLOCATION', 
            fontsize=20, fontweight='bold')
plt.tight_layout()
plt.show()

# Print key insights
print("\n" + "=" * 70)
print("KEY INSIGHTS: LIFECYCLE INVESTING")
print("=" * 70)

print("\n1. ALLOCATION STRATEGIES AT AGE 30:")
idx_30 = np.where(ages == 30)[0][0]
print(f"   Simple Rule (100-age):      {allocation_simple[idx_30]*100:.1f}%")
print(f"   Time-Varying Risk Aversion: {allocation_time_varying_gamma[idx_30]*100:.1f}%")
print(f"   With Human Capital:         {allocation_with_hc[idx_30]*100:.1f}%")
print(f"   Target Date Fund:           {allocation_target_date[idx_30]*100:.1f}%")

print("\n2. ALLOCATION STRATEGIES AT AGE 60:")
idx_60 = np.where(ages == 60)[0][0]
print(f"   Simple Rule (100-age):      {allocation_simple[idx_60]*100:.1f}%")
print(f"   Time-Varying Risk Aversion: {allocation_time_varying_gamma[idx_60]*100:.1f}%")
print(f"   With Human Capital:         {allocation_with_hc[idx_60]*100:.1f}%")
print(f"   Target Date Fund:           {allocation_target_date[idx_60]*100:.1f}%")

print("\n3. HUMAN CAPITAL INSIGHTS:")
print(f"   Age 25: Human capital = ${human_capital[0]:,.0f} ({hc_fraction[0]*100:.1f}% of total)")
print(f"   Age 40: Human capital = ${human_capital[15]:,.0f} ({hc_fraction[15]*100:.1f}% of total)")
print(f"   Age 60: Human capital = ${human_capital[35]:,.0f} ({hc_fraction[35]*100:.1f}% of total)")

print("\n4. TERMINAL WEALTH (Age 65) BY STRATEGY:")
for strategy, W in wealth_paths.items():
    print(f"   {strategy:25s}: ${W[-len(W)-retirement_age+69]:,.0f}")

print("\n5. PRACTICAL RECOMMENDATIONS:")
print("   ✓ Young investors (20s-30s): Can be MORE aggressive (100%+ stocks)")
print("      - Large human capital acts as bond holding")
print("      - More time to recover from losses")
print("      - Higher earnings growth potential")
print("\n   ✓ Middle age (40s-50s): Moderate allocation (60-80% stocks)")
print("      - Human capital declining")
print("      - Peak earning years")
print("      - Moderate time horizon")
print("\n   ✓ Near retirement (60+): Conservative (40-60% stocks)")
print("      - Little human capital remaining")
print("      - Limited time to recover losses")
print("      - Need for stable income")

print("\n6. COMMON MISTAKES:")
print("   ✗ Being too conservative when young")
print("   ✗ Ignoring human capital in portfolio decisions")
print("   ✗ Not adjusting allocation as you age")
print("   ✗ Panic selling in downturns")
print("   ✗ Chasing returns near retirement")

print("\n7. EXTENSIONS:")
print("   • Stochastic labor income (unemployment risk)")
print("   • Correlation between labor income and stock returns")
print("   • Flexible retirement date (option value)")
print("   • Health shocks and medical expenses")
print("   • Bequest motives")

print("\n" + "=" * 70)
print("Lifecycle investing analysis complete!")
print("=" * 70)

## 4. Visualization and Analysis

### 4.1 Wealth and Consumption Paths Under Optimal Control

## 5. Practical Applications

## Summary\n\nThis notebook covered stochastic control for asset allocation. Key takeaways:\n\n- Practical implementation with Python\n- Visualizations and interpretations\n- Real-world applications\n- Best practices and pitfalls\n\n### Next Steps\n\n- Practice with real market data\n- Explore related topics\n- Build your own variations\n\n### Additional Resources\n\n- [QuantEdX Courses](https://www.quantedx.com/courses)\n- [Community Forum](https://www.quantedx.com/community)\n- [GitHub Repository](https://github.com/quantedx)